# Project Icarus - Engagement Sweep Notebook

This notebook demonstrates the end-to-end simulation API for interceptor engagements.

## 1. Imports

In [ ]:
import sys
import os

# Ensure project root is on sys.path regardless of notebook location
possible_roots = [
    os.path.abspath(os.path.join(os.getcwd(), "..")),
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
for root in possible_roots:
    if os.path.isdir(os.path.join(root, "src")):
        if root not in sys.path:
            sys.path.insert(0, root)
        break

import numpy as np
import matplotlib.pyplot as plt

from project_icarus.interceptors.config import InterceptorConfig
from project_icarus.guidance.law import GuidanceLaw
from project_icarus.scenarios.target_factory import (
    BallisticScenario,
    FOBSScenario,
    HGVScenario,
    SuppressedScenario,
    SwarmScenario,
)
from project_icarus.scenarios.scenario import EngagementScenario
from project_icarus.sim.api import run_engagement, run_sweep

print("Imports successful.")

## 2. Define System Configurations

In [ ]:
interceptor = InterceptorConfig(
    name="Arrow 3",
    mass=1000.0,
    area=0.1,
    ref_length=1.0,
    kill_radius=0.5,
    kill_mechanism="hit_to_kill",
    mkv_divert_impulse=85.0,
)

guidance = GuidanceLaw()

scenario = EngagementScenario(
    name="Default",
    interceptor_launch_site=np.array([6371e3, 0.0, 0.0]),
    engagement_start=0.0,
    engagement_end=300.0,
)

## 3. Single Engagement: Ballistic Target

In [ ]:
target_ballistic = BallisticScenario(
    r0=np.array([6371e3, 0.0, 0.0]),
    v0=np.array([0.0, 1000.0, 1000.0]),
)

result = run_engagement(interceptor, guidance, target_ballistic, scenario, n_trials=30)
print(f"Nominal miss: {result.miss_distance:.2f} m")
print(f"Kill: {result.kill_assessment}")
if result.monte_carlo:
    print(f"MC kill probability: {result.monte_carlo.kill_probability:.2%}")

## 4. Single Engagement: FOBS Target

In [ ]:
target_fobs = FOBSScenario.from_orbital_params(
    apoapsis_km=200.0,
    inclination_deg=30.0,
)

result_fobs = run_engagement(interceptor, guidance, target_fobs, scenario, n_trials=30)
print(f"FOBS nominal miss: {result_fobs.miss_distance:.2f} m")
print(f"FOBS kill: {result_fobs.kill_assessment}")

## 5. Batch Sweep

In [ ]:
targets = [
    BallisticScenario(r0=np.array([6371e3, 0.0, 0.0]), v0=np.array([0.0, 1000.0, 1000.0])),
    HGVScenario.from_params(max_alt_km=80.0, lateral_range_km=2000.0),
    SuppressedScenario(r0=np.array([6371e3, 0.0, 0.0]), v0=np.array([0.0, 800.0, 800.0])),
]
scenarios = [scenario]

sweep_result = run_sweep(
    interceptors=[interceptor],
    targets=targets,
    scenarios=scenarios,
    n_trials=20,
)

df = sweep_result.to_dataframe()
if df is not None:
    display(df)

## 6. Visualization

In [ ]:
if result.monte_carlo:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    result.plot_miss_distance_distribution(ax=axes[0])
    axes[0].set_title("Miss Distance Distribution")
    
    result.plot_3d(ax=axes[1])
    axes[1].set_title("3D Trajectory")
    plt.tight_layout()
    plt.show()